# Zong et al. 2022 analysis pipeline on our MINI2P data

Reproduces the neuronal-activity pipeline of **Zong, W. et al. (2022). Large-scale two-photon calcium imaging in freely moving mice. *Cell* 185, 1240-1256** (STAR Methods: "Estimation of motion, and motion correction", "Neuronal activity processing", and the "escape-active cells" shuffle test), applied to our reward-aligned imaging runs. It is a separate notebook from `reward_responses.ipynb`: it starts from the **registered planes and Cellpose masks** (registration and cell detection done; extraction not), extracts the traces itself and never reads or overwrites the plane's own `F.npy`, `Fneu.npy`, `stat.npy`.

| Section | Paper step | Code |
|---|---|---|
| 1 | Cellpose ROIs aligned onto each variant; `Fcell`, `Fnp` (weighted averages in/around each ROI, excluding pixels of other ROIs), both channels | `zong2022_io`, mini2p `build_stat`, Suite2p `create_masks` |
| 2 | `Fcorr = Fcell - 0.7 Fnp`; running baseline `F0 = Fs + m` (Low et al. 2014); `ΔF/F`; significant transients; OASIS deconvolution; `ΔF/F_clear`, `E_clear`, calcium events | `zong2022_signal` |
| 3 | SNR per cell, inclusion (SNR > 3) | `zong2022_signal` |
| 4 | "Metrics for registration": 30 spatial PCs, Itop/Ibottom, residual xy drift, z-drift review | `zong2022_motion` (Suite2p's `pclowhigh`, `pc_register`) |
| 5 | Reward-aligned responses by number of pulses | `zong2022_responses` |
| 6 | Responsive cells: score and 1000 circular time shifts | `zong2022_responses` |

Cache: extracted traces, the signal processing and the registration metrics are cached in the repo's git-ignored `.cache/zong2022/`, keyed on file sizes and modification times, the settings, the suite2p and mini2p versions and the code of the modules that affect each stage (a new library version or an edited module recomputes). Nothing is written to the data share unless `WRITE_DERIVATIVES` (then to `<run>/<variant>/zong2022/`, never into `suite2p/`).

## Where we follow the paper and where we had to decide

The methods text is brief at several steps. Every choice is below; each is a setting or a short function that is easy to change.

- **Indicator.** The paper used GCaMP6s (decay 1.5 s). Ours is **jGCaMP8s**. `TAU_S = 0.7` s is a placeholder, not a measured value: the jGCaMP8s half-decay after one action potential is about 0.31 s in cultured neurons (Zhang et al., *Nature* 2023, jGCaMP8 characterisation), i.e. an exponential τ of about 0.44 s, and decay in vivo is slower. **Set `TAU_S` from your own data or the in-vivo literature.** Section 2 compares event counts for τ = 0.45, 0.7 and 1.0 s so you can see how much it matters.
- **What OASIS is given.** The paper deconvolves `Fcorr`. OASIS has no baseline term, so a trace sitting at `F0` would be explained by one huge event; we deconvolve `ΔF = Fcorr - F0` (in F units), as Suite2p and mini2p do. This is also the purpose of the constant `m`: it centres `Fcorr - F0` on zero at baseline points.
- **`F0` and the constant `m`.** `Fs` is the 8th percentile of `Fcorr` in a ±15 s window centred on each frame (cut to the frames that exist within 15 s of the recording's ends). Baseline points are frames whose local SD (same window) is at most `sd_min + 0.1 (sd_max - sd_min)`, over all frames of the cell. `m` is the mean of `Fcorr - Fs` over the baseline points, so `Fcorr - F0` has mean zero there.
- **"Local standard deviation of the baseline fluorescence".** For the transient threshold we use a robust SD, `1.4826 × MAD` of `ΔF/F`, over the frames of the ±15 s window of each frame that are *not in a significant transient*, in two passes: first with a threshold from all frames, then again excluding the transients found. Frames of a transient do not raise their own threshold, and the MAD is barely moved by the few that slip through. A window with fewer than `MIN_BASELINE_S` (4 s) of such frames uses the cell's overall value. The SD is evaluated on a coarse grid and interpolated, which changes it negligibly.
- **Cells with too little baseline.** `m`, `F0` and the noise level need baseline points. A cell with fewer than `MIN_BASELINE_S` (4 s) of baseline-point frames, or no non-transient frames, is excluded with that reason (never an error) and listed in section 3. Which cells these are depends on the data, not the code: a cell that is active nearly all the time has no baseline.
- **Limitation: correlated noise.** The 2 SD / 0.75 s rule assumes noise that is roughly white at the frame rate. Synthetic cells with no transient at all, run through the pipeline (`tests/test_zong2022_signal.py`, `noise_pass_rates`), pass the SNR > 3 inclusion in 0 % of cases with white noise, 4 % with AR(1) noise of τ = 0.2 s and **85 % with τ = 0.5 s**: slowly varying noise makes runs of 6 frames above 2 SD common, and for such smooth noise the frame-to-frame difference (the SNR denominator) is small. If your traces have slow, noise-like structure at 0.5 s (motion, indicator kinetics, residual neuropil), SNR > 3 does not by itself show real transients; look at the example traces. We report this and do not change the paper's rule.
- **Baseline points near the recording's ends.** The ±15 s window is cut, not padded, at the ends, so a frame within 15 s of an end sees a window of 15 to 30 s. Its local SD is estimated from fewer frames and so is more variable, which makes edge windows likelier to fall in the lowest tenth of the SD range: edge frames are somewhat over-represented among baseline points. `m` is a mean over hundreds to thousands of points, so the effect is small, but it is not zero.
- **Significant transients.** `ΔF/F` above 2 local baseline SDs for *more than* 0.75 s: a run of `k` frames counts when `k × frame period > 0.75 s` (6 frames at 7.5 Hz).
- **SNR.** Signal = mean over significant transients of the 90th percentile of `ΔF/F` within each transient. Noise = mean absolute difference between consecutive frames of `ΔF/F`, over pairs of frames that both lie outside significant transients (the paper says "mean of differences of DF/F in periods outside significant transients"; the absolute value is needed for a mean difference to be positive). A cell with no transient or zero noise has no SNR and does not pass. Cells need SNR > 3; `MIN_EVENTS` (the paper used 100 for its tuning analyses) is off by default.
- **Cells with `F0 <= 0`.** The paper does not say what to do with a cell whose baseline is not positive (ΔF/F is meaningless). Our choice: the cell is excluded from everything downstream and listed in section 3 with the reason. Rows keep their numbering. A run in which no cell is usable is left out with a message.
- **ROI numbers.** Aligning the masks can drop whole ROIs, so the Cellpose labels have gaps. After alignment the labels are renumbered 1..n in increasing Cellpose order; the table's `roi` is that count (0-based, the row of the traces) and its `cellpose_label` column, the figures' titles and the numbers on the field of view, and the saved `traces.npz` (`cellpose_label`) carry the original Cellpose label.
- **Overlapping pixels.** Suite2p's `create_masks` defaults are used (weights normalised, pixels flagged as overlapping another ROI removed, neuropil = surrounding ring without any ROI's pixels). Cellpose labels cannot overlap, so no pixel is removed in practice.
- **Cells** are all Cellpose ROIs; Suite2p's `iscell` classifier is not used.
- **Registration metrics.** Suite2p's own `pclowhigh` and `pc_register` are used, with 500 frames per extreme image (the paper; Suite2p's GUI uses 300; fewer if the movie is shorter than 1000 sampled frames) from at most `MAX_PC_SOURCE_FRAMES` (default 2000; the paper used 5000) evenly sampled frames of the registered green movie, cut to the registration's valid crop. **Memory:** the PCA holds the sampled frames in float32 and several working copies, so its peak is about 4 GB at 256 × 256 pixels and 16 GB at 512 × 512 for 5000 frames, in proportion to the number of frames; the 2000 default needs about 40 % of that. Raise it only on a machine with the memory. Suite2p registers Itop to Ibottom (rigid, then non-rigid blocks); the paper registers both to the stack mean. Both measure a shift between the extreme frames that motion correction should have removed. Mean and maximum are over blocks.
- **Responsive cells.** The paper's escape-active score `(A_event - A_other) / (A_event + A_other)` is computed here for rewarded outcomes and, separately, for no-reward outcomes. `A` is the cell's mean `ΔF/F_clear` (`SCORE_TRACE`) over frames in the 0-2 s window after the outcome's frame (`RESPONSE_S`), across all those outcomes, against the mean over all other frames of the recording. The null is 1000 circular shifts of the cell's *scored trace* (`ΔF/F_clear`), each at least 30 s from either end, wrapped. The paper shifts the deconvolved activity; we shift the trace that is scored so that the null keeps exactly the temporal structure of what is compared (transient shapes, durations, spacing), whereas shifting an event train and scoring a different trace would compare two different signals. a cell is responsive when its score exceeds the 95th percentile of its own shifts. `ΔF/F_clear` is never negative, so the score lies between -1 and 1; with raw `ΔF/F` the denominator could be near zero. Cells without a finite score (excluded cells) are not tested and are left out of the counts against chance. Only outcomes whose whole −2 to 5 s window lies on imaged frames are used, so the figures and the test describe the same trials.
- **Slow signals are removed by design.** The 30 s baseline, the transient filter and the deconvolution discard slow, field-wide fluctuations, unlike the robust-z views of `reward_responses.ipynb`; a response that is slow or shared across the field will be smaller or absent here.

Needs `uv sync --extra funcimg --extra notebooks`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib.backends.backend_pdf import PdfPages

import src.behavior_visualization.plot_style  # noqa: F401
from src.behavior_analysis.outcome_events import load_outcome_events
from src.funcimg_analysis.zong2022_io import write_derivatives, write_table
from src.funcimg_analysis.zong2022_motion import MotionSettings, nonrigid_motion
from src.funcimg_analysis.zong2022_responses import ResponseSettings
from src.funcimg_analysis.zong2022_run import (
    analyse_runs,
    derivative_arrays,
    pool_reward_runs,
    top_responsive_cells,
)
from src.funcimg_analysis.zong2022_signal import Zong2022Settings, event_counts_by_tau
from src.funcimg_visualization.plot_funcimg_zong2022 import (
    plot_baseline_example,
    plot_clear_responses,
    plot_motion_summary,
    plot_score_vs_null,
    plot_signal_examples,
    plot_snr_histogram,
    plot_tau_sensitivity,
    plot_top_responsive,
    plot_zdrift,
    zong2022_png_name,
)
from src.settings import REPO_ROOT, get_settings

EXPERIMENT = "3x3_field_magnitude_bandit"
COHORT = "cohort-01"
SUBJECT = "sub-03_id-MY_71_R"
SESSION_FIRST, SESSION_LAST = 224, 224  # one session: give the same number twice
RUNS = ("run-01",)  # every run: None; or only these runs of each session
VARIANTS = ("without_destriping",)  # add "with_destriping" to compare

# Signal processing: the paper's values, except TAU_S (see section 0).
NEUROPIL_COEFFICIENT = 0.7
TAU_S = 0.7  # jGCaMP8s placeholder; the paper's 1.5 s is for GCaMP6s
BASELINE_PERCENTILE = 8
BASELINE_HALF_WINDOW_S = 15.0
BASELINE_SD_FRACTION = 0.1
TRANSIENT_SD = 2.0
TRANSIENT_MIN_S = 0.75
MIN_SNR = 3.0
MIN_EVENTS = None  # the paper used 100 for its tuning analyses
MIN_BASELINE_S = 4.0  # seconds of baseline points a cell needs; fewer: excluded with a reason
TAU_SENSITIVITY_S = (0.45, 0.7, 1.0)

# Reward analysis
WINDOW_S = (-2.0, 5.0)  # around the outcome, for the average responses
RESPONSE_S = 2.0  # post-outcome window for the responsive-cell score
N_SHIFTS = 1000
MIN_SHIFT_S = 30.0
SCORE_TRACE = "dff_clear"  # or "e_clear"
PHASES = ("task",)  # add "port_finding" to include port-finding trials
X_UNITS = "s"  # "s" or "frames"
N_TOP = 12  # most responsive cells drawn per run and outcome kind
SEED = 0

# Motion and z drift
N_PCS = 30
N_PC_FRAMES = 500  # frames averaged for each of Itop and Ibottom
MAX_PC_SOURCE_FRAMES = 2000  # the paper used 5000; memory grows with it (about 4 GB at 256 x 256)
N_ZDRIFT_PCS_SHOWN = 6  # panels on screen; the PDF has all N_PCS

WRITE_DERIVATIVES = False  # True: traces to <run>/<variant>/zong2022/ (never into suite2p/)
OUTCOMES_LOADER = load_outcome_events

ZONG = Zong2022Settings(
    neuropil_coefficient=NEUROPIL_COEFFICIENT, tau_s=TAU_S,
    baseline_percentile=BASELINE_PERCENTILE, baseline_half_window_s=BASELINE_HALF_WINDOW_S,
    baseline_sd_fraction=BASELINE_SD_FRACTION, transient_sd=TRANSIENT_SD,
    transient_min_s=TRANSIENT_MIN_S, min_snr=MIN_SNR, min_events=MIN_EVENTS,
    min_baseline_s=MIN_BASELINE_S,
)
RESPONSES = ResponseSettings(
    window_s=WINDOW_S, response_s=RESPONSE_S, n_shifts=N_SHIFTS, min_shift_s=MIN_SHIFT_S,
    score_trace=SCORE_TRACE, phases=PHASES, seed=SEED,
)
MOTION = MotionSettings(n_pcs=N_PCS, n_extremes=N_PC_FRAMES, max_frames=MAX_PC_SOURCE_FRAMES)

data_root = get_settings().data_root
DERIVATIVES = data_root / EXPERIMENT / "derivatives" / COHORT / SUBJECT
FIGURES = REPO_ROOT / "results" / "figures" / "funcimg" / COHORT / SUBJECT / "zong2022"
FIGURES.mkdir(parents=True, exist_ok=True)
CACHE_DIR = get_settings().cache_dir / "zong2022"
SCOPE = f"sessions {SESSION_FIRST}-{SESSION_LAST}" + (f", {', '.join(RUNS)}" if RUNS else "")
FILE_TAG = f"ses-{SESSION_FIRST}-{SESSION_LAST}" + (f"_{'_'.join(RUNS)}" if RUNS else "")


def run_tag(run):
    return f"{run['session'].split('_')[0]}_{run['run']}"

## 1. Masks and extraction

For each run the Cellpose masks drawn on `<run>/suite2p/plane0` are shifted onto the variant's own mean image (mini2p `align_to_plane`; the shift and the rank correlation after it are in the table) and `Fcell`, `Fnp` are extracted from the registered green and red binaries with Suite2p's masks. `Fcell` and `Fnp` are weighted averages within and around each ROI, excluding pixels that overlap other ROIs. The registered movies are read in chunks; the result is cached, so a re-run takes seconds. Runs left out (no frame times, behaviour file or segmentation, a refused mask alignment, no usable cell, traces from another acquisition) are listed with the reason. The ROI diameter for the alignment is mini2p's `RegistrationSettings.diameter_px`. The `ROIs` column counts table rows; the Cellpose labels of the rows are in each table's `cellpose_label` column. Run `reward_responses.ipynb` section 1 first if a run has no `frame_times.npz`.

In [ ]:
results = {}
for variant in VARIANTS:
    runs, skipped = analyse_runs(
        DERIVATIVES, SESSION_FIRST, SESSION_LAST, variant, RUNS,
        settings=ZONG, responses=RESPONSES, motion=MOTION, cache_dir=CACHE_DIR,
        load_outcomes=OUTCOMES_LOADER, report=print,
    )
    results[variant] = runs
    for session, run, reason in skipped:
        print(f"  LEFT OUT {variant} {session} {run}: {reason}")
    display(pd.DataFrame([{
        "session": r["session"], "run": r["run"], "ROIs": len(r["table"]), "frames": r["n_frames"],
        "mask shift dy,dx": f"{r['alignment']['dy']},{r['alignment']['dx']}",
        "mask r after": round(r["alignment"]["rank_correlation_after"], 2),
        "masks dropped": len(r["alignment"]["dropped_labels"]),
        "frame period (s)": round(r["period_s"], 4),
    } for r in runs]))

## 2. ΔF/F, significant transients and calcium events

`Fcorr = Fcell - 0.7 Fnp`; running baseline `F0`; `ΔF/F = (Fcorr - F0) / F0`; significant transients; OASIS events. The first figure shows one cell's `Fcorr`, its baseline `F0` and the baseline points (frames in the lowest tenth of the local-SD range). The second shows three cells' `ΔF/F` (grey), `ΔF/F_clear` (blue, zero outside significant transients) and `E_clear` (red; events scaled so their maximum equals the maximum of `ΔF/F_clear`, so both share the axis). "Calcium events" are the non-zero frames of `E_clear`.

**Sensitivity to τ.** The last cell deconvolves the same baseline-subtracted traces with three decay times and counts events inside significant transients. A larger τ explains each transient's tail with fewer events. `TAU_S` is the bar in blue. If the count changes a lot across the plausible range for jGCaMP8s, the event-based results (`E_clear`) depend on that choice; `ΔF/F_clear` does not.

In [ ]:
for variant, runs in results.items():
    if not runs:
        print(f"{variant}: no run to show")
        continue
    run = runs[0]
    signal, table = run["signal"], run["table"]
    scored = table[table["usable"] & table["snr"].notna()].sort_values("snr", ascending=False)
    ranked = scored["roi"].to_numpy()[:3]
    if not len(ranked):
        print(f"{variant} {run['session']} {run['run']}: no cell has a transient to show; skipped")
        continue
    labels_shown = scored["cellpose_label"].to_numpy()[:3].tolist()
    print(f"{variant} {run['session']} {run['run']}: example ROIs {ranked.tolist()} "
          f"(Cellpose labels {labels_shown}; highest SNR)")
    fig = plot_baseline_example(signal, int(ranked[0]))
    plt.show()
    fig = plot_signal_examples(
        signal, ranked, (0.0, min(300.0, run["n_frames"] * run["period_s"])),
        title=f"{SUBJECT}, {run['session'].split('_')[0]} {run['run']}, {variant}",
    )
    fig.savefig(FIGURES / zong2022_png_name("signal_examples", run_tag(run), variant),
                dpi=200)
    plt.show()

In [ ]:
for variant, runs in results.items():
    if not runs:
        continue
    per_tau = {}
    for run in runs:
        signal = run["signal"]
        counts = event_counts_by_tau(
            signal.delta_f, signal.transient_mask, 1 / run["period_s"],
            tuple(sorted({*TAU_SENSITIVITY_S, TAU_S})),
        )
        for tau, per_cell in counts.items():
            per_tau.setdefault(tau, []).append(per_cell[signal.usable])
    pooled = {tau: np.concatenate(parts) for tau, parts in per_tau.items()}
    fig = plot_tau_sensitivity(pooled, TAU_S, title=f"{variant}: events per cell against τ")
    fig.savefig(FIGURES / zong2022_png_name("tau_sensitivity", FILE_TAG, variant), dpi=200)
    plt.show()
    display(pd.DataFrame({"tau (s)": list(pooled),
                          "mean events per cell": [v.mean().round(1) for v in pooled.values()]}))

## 3. SNR and inclusion

Signal is the mean over a cell's significant transients of the 90th percentile of `ΔF/F` within each; noise is the mean absolute frame-to-frame difference of `ΔF/F` outside transients. Cells need SNR > `MIN_SNR` (and at least `MIN_EVENTS` calcium events when set). The tables list per cell the SNR, transient and event counts, inclusion, and the responsive-cell results of section 6; they are saved as CSV next to the figures. Excluded cells (`F0 <= 0` or too few baseline points) are listed by Cellpose label with the reason (our choice, see section 0).

In [ ]:
for variant, runs in results.items():
    rows = []
    for run in runs:
        table = run["table"]
        out = table[~table["usable"]]
        excluded = dict(zip(out["cellpose_label"], out["excluded"], strict=True))
        rows.append({
            "session": run["session"], "run": run["run"], "ROIs": len(table),
            "excluded (Cellpose label: reason)": str(excluded) if excluded else "-",
            "no transient": int((table["usable"] & (table["n_transients"] == 0)).sum()),
            f"SNR > {MIN_SNR:g}": int(table["passes"].sum()),
            "median SNR": round(float(table["snr"].median()), 1),
            "median events (passing)": float(table.loc[table["passes"], "n_events"].median()),
        })
        path = write_table(
            table, FIGURES / f"cell_table_{run_tag(run)}_{variant}.csv"
        )
        print("Saved", path)
    display(pd.DataFrame(rows))
    if runs:
        fig = plot_snr_histogram(
            pd.concat([r["table"] for r in runs], ignore_index=True), MIN_SNR,
            title=f"{variant}: SNR of all cells",
        )
        fig.savefig(FIGURES / zong2022_png_name("snr", FILE_TAG, variant), dpi=200)
        plt.show()

In [ ]:
if WRITE_DERIVATIVES:
    for variant, runs in results.items():
        for run in runs:
            out = write_derivatives(
                run["run_dir"] / variant,
                derivative_arrays(run),  # includes each row's Cellpose label
                {"zong2022": ZONG.__dict__, "responses": RESPONSES.__dict__},
            )
            print("Wrote", out)

## 4. Motion correction and z drift

Suite2p's "Metrics for registration" as the paper uses it: 30 spatial PCs of up to `MAX_PC_SOURCE_FRAMES` (2000 by default; the paper used 5000, see the memory note in section 0) evenly sampled registered frames; for each PC, **Itop** and **Ibottom** are the means of the 500 highest- and 500 lowest-scoring frames. Their residual xy shift (Itop registered to Ibottom, rigid then non-rigid) is what motion correction left behind: **low mean and maximum drift in all PCs mean those PCs are not explained by xy motion**. The summary figure also shows the rigid shifts (`xoff`, `yoff`) and the paper's non-rigid motion (the largest distance between any two blocks' shifts in a frame, from `xoff1`, `yoff1`) over the recording, when `ops.npy` has them.

**z drift** is judged by eye (Stringer & Pachitariu 2019): in `Ibottom - Itop`, a cell's nucleus and cytoplasm should brighten and dim together. If one is white and the other black, the difference cannot be calcium and indicates z drift. Each PC is drawn twice: a plain panel and a panel with a thin contour just outside each ROI (so the ROI's own pixels stay visible). The first `N_ZDRIFT_PCS_SHOWN` PCs are shown; the PDF has all of them, six per page.

In [ ]:
for variant, runs in results.items():
    for run in runs:
        motion = run["motion"]
        offsets = motion["offsets"]
        nonrigid = (
            nonrigid_motion(offsets["xoff1"], offsets["yoff1"])
            if {"xoff1", "yoff1"} <= set(offsets) else None
        )
        title = f"{SUBJECT}, {run['session'].split('_')[0]} {run['run']}, {variant}"
        drift = motion["drift"]
        print(f"{title}: {motion['n_frames_sampled']} frames sampled, "
              f"{motion['n_extremes']} per extreme image")
        print(f"  residual drift over {len(drift)} PCs: mean-over-blocks max "
              f"{drift['mean_nonrigid_px'].max():.2f} px, max-over-blocks max "
              f"{drift['max_nonrigid_px'].max():.2f} px, "
              f"rigid max {drift['rigid_px'].max():.2f} px")
        fig = plot_motion_summary(drift, offsets, nonrigid, run["period_s"], title=title)
        fig.savefig(FIGURES / zong2022_png_name("motion", run_tag(run), variant),
                    dpi=200)
        plt.show()
        y0, y1, x0, x1 = motion["crop"]
        labels = run["labels"][y0:y1, x0:x1]
        n_pcs = len(drift)
        pdf_path = FIGURES / f"zdrift_{run_tag(run)}_{variant}.pdf"
        with PdfPages(pdf_path) as pdf:
            for start in range(0, n_pcs, 6):
                page = plot_zdrift(
                    motion["i_top"], motion["i_bottom"], labels,
                    range(start, min(start + 6, n_pcs)), title=title,
                )
                pdf.savefig(page)
                plt.close(page)
        print("Saved", pdf_path)
        fig = plot_zdrift(
            motion["i_top"], motion["i_bottom"], labels,
            range(min(N_ZDRIFT_PCS_SHOWN, n_pcs)), title=title,
        )
        fig.savefig(FIGURES / zong2022_png_name("zdrift", run_tag(run), variant),
                    dpi=150)
        plt.show()

## 5. Reward-aligned responses

Mean ± SEM across cells of each cell's trial-averaged `ΔF/F_clear` and `E_clear`, by number of reward pulses (0 at the no-reward cue, 1-4 at reward onset, as in `reward_responses.ipynb`), from `WINDOW_S` = −2 to 5 s around the outcome's frame. Left: cells above the SNR threshold, as the paper selects cells for its tuning analyses; right: all cells with a defined `ΔF/F`, for comparison. Values are zero outside significant transients by construction and nothing is baseline-subtracted. Each run's cells count once; cells of different runs of one field may be the same neurons.

In [ ]:
for variant, runs in results.items():
    if not runs:
        continue
    pooled = {
        trace: {group: pool_reward_runs(runs, trace, group) for group in ("snr_pass", "all")}
        for trace in ("dff_clear", "e_clear")
    }
    first = runs[0]["reward"]["aligned"]["dff_clear"]
    display(pd.DataFrame([{
        "session": r["session"], "run": r["run"],
        **{f"{p} pulses": r["reward"]["n_trials"].get(p, 0) for p in range(5)},
        "outcomes dropped": r["reward"]["n_dropped"],
    } for r in runs]))
    fig = plot_clear_responses(
        pooled, first.offsets, first.period_s, x_units=X_UNITS,
        title=f"{SUBJECT}, {SCOPE}, {variant}: mean ± SEM across cells",
    )
    fig.savefig(FIGURES / zong2022_png_name("clear_responses", FILE_TAG, f"{variant}_{X_UNITS}"), dpi=200)
    plt.show()

## 6. Responsive cells

The paper's escape-active test, applied to outcomes: each cell's score `(A_reward - A_other) / (A_reward + A_other)` against the scores of 1000 circular time shifts of its own activity (at least 30 s from either end, wrapped); a cell is responsive when its score exceeds the 95th percentile of its shifts. Done separately for rewarded outcomes (1-4 pulses) and no-reward outcomes (0 pulses); see section 0 for the exact definition of `A`. About 5 % of cells would pass if nothing responded, so the table gives the observed count beside the expected one and a rough binomial p (cells are not independent, so treat it as a guide). The score is a ratio of means over very different numbers of frames, so a cell with few events can score high by luck; the shifted null accounts for that.

The figures show, per run and kind, each cell's score against its own null 95th percentile, and the `N_TOP` highest-scoring cells that are responsive and above the SNR threshold: `ΔF/F_clear` around the outcome by pulse count (mean ± SEM across trials), SNR and score in the title, and the cells outlined and numbered (by Cellpose label) on the mean image. In the table, "tested" counts the cells with a score; cells without one (excluded) are not part of the chance comparison.

In [ ]:
KINDS = ("reward", "no_reward")
SUMMARIES = (("all cells", "summary_all"), (f"SNR > {MIN_SNR:g}", "summary_pass"))
rows = []
for runs in results.values():
    for run in runs:
        for kind in KINDS:
            test = run["reward"]["tests"][kind]
            if test is None:
                print(f"  {run['session']} {run['run']}: no {kind.replace('_', ' ')} outcomes")
                continue
            for group, key in SUMMARIES:
                s = test[key]
                rows.append({
                    "session": run["session"], "run": run["run"], "outcomes": kind, "group": group,
                    "cells": s["n_cells"], "tested": s["n_tested"],
                    "responsive": s["n_responsive"],
                    "expected by chance": round(s["expected_by_chance"], 1),
                    "binomial p": round(s["p_binomial"], 4),
                })
display(pd.DataFrame(rows))

In [ ]:
for variant, runs in results.items():
    for run in runs:
        table, aligned = run["table"], run["reward"]["aligned"]["dff_clear"]
        title = f"{SUBJECT}, {run['session'].split('_')[0]} {run['run']}, {variant}"
        for kind in KINDS:
            if run["reward"]["tests"][kind] is None:
                continue
            fig = plot_score_vs_null(table, kind, title=f"{title}: {kind.replace('_', ' ')}")
            name = zong2022_png_name(f"score_{kind}", run_tag(run), variant)
            fig.savefig(FIGURES / name, dpi=200)
            plt.show()
            top = top_responsive_cells(table, kind, N_TOP)
            if not len(top):
                print(f"  {title}: no responsive {kind} cell above SNR {MIN_SNR:g}")
                continue
            fig = plot_top_responsive(
                aligned.windows, aligned.pulses, top, table, kind, aligned.offsets,
                aligned.period_s, run["mean_image"], run["labels"], x_units=X_UNITS,
                title=f"{title}: most {kind.replace('_', '-')}-responsive cells",
            )
            name = zong2022_png_name(f"top_responsive_{kind}", run_tag(run), variant)
            fig.savefig(FIGURES / name, dpi=200)
            plt.show()